In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# 1. 矿物组成（质量分数）
mineral_fractions = {'Ol': 0.60, 'Opx': 0.25, 'Cpx': 0.10, 'Gar': 0.05}

# 2. 稀土元素序列
ree_elements = ['La', 'Ce', 'Nd', 'Sm', 'Eu', 'Dy', 'Er', 'Yb', 'Lu']

# 3. 分配系数矩阵 (Kd_data)
kd_data = {
    'Ol':  [0.007, 0.006, 0.006, 0.007, 0.007, 0.013, 0.026, 0.049, 0.045],
    'Opx': [0.030, 0.020, 0.030, 0.050, 0.050, 0.150, 0.230, 0.340, 0.420],
    'Cpx': [0.056, 0.092, 0.230, 0.445, 0.474, 0.582, 0.583, 0.542, 0.506],
    'Gar': [0.001, 0.007, 0.026, 0.102, 0.243, 1.940, 4.700, 6.167, 6.950]
}

# 构建 DataFrame，行索引为矿物名称，列索引为稀土元素
Kd_df = pd.DataFrame.from_dict(kd_data, orient='index', columns=ree_elements)

# 4. 熔融程度
F_values = [0.02, 0.1]

print("初始化完成，分配系数矩阵如下：")
display(Kd_df) if 'display' in globals() else print(Kd_df)

In [ ]:
# 利用 pandas 的运算将 Kd_df 乘以对应的矿物比例，然后按列求和
fraction_series = pd.Series(mineral_fractions)
bulk_D = Kd_df.mul(fraction_series, axis=0).sum(axis=0)

print("各稀土元素的总分配系数 (Bulk D):")
print(bulk_D)

In [ ]:
# 定义批式熔融计算函数
def calc_batch_melting(bulk_D, F):
    # C_l / C_0 = 1 / ((1 - F) * D + F)
    return 1 / ((1 - F) * bulk_D + F)

# 循环计算不同熔融程度下的结果
melt_compositions = {}
for F in F_values:
    # 存入字典
    melt_compositions[f'F={F}'] = calc_batch_melting(bulk_D, F)

# 转换为 DataFrame 方便查看和绘图
melt_df = pd.DataFrame(melt_compositions)

print("不同批式熔融程度下，熔体相对球粒陨石的富集系数:")
display(melt_df) if 'display' in globals() else print(melt_df)

In [ ]:
# 创建绘图画布，设置图幅大小
fig, ax = plt.subplots(figsize=(10, 6))

# 循环遍历 melt_compositions 字典，绘制折线图
for label_name, composition in melt_compositions.items():
    ax.plot(ree_elements, composition, marker='o', label=label_name)

# Y轴设置为对数坐标轴
ax.set_yscale('log')

# 设置轴标签和标题
ax.set_xlabel('Rare Earth Elements', fontsize=12)
ax.set_ylabel('Rock / Chondrite (Log Scale)', fontsize=12)
ax.set_title('REE Patterns of Batch Melting for Garnet Lherzolite', fontsize=14)

# 添加图例
ax.legend(fontsize=12)

# 开启网格线（Y轴方向同时显示 major 和 minor 网格）
ax.grid(True, which='both', linestyle='--', linewidth=0.5, alpha=0.7)

# 显示图形
plt.show()